# Introduction
This colab worksheet provides a starting point for Task 1 (the natural language processing assignment).

# Data Loading

In [ ]:
# Download the data stored in a csv file from one of these two locations
# If you're running all your experiments
# on a machine at home rather than using colab, then make sure you save it
# rather than repeatedly downloading it.

!wget "https://sussex.box.com/shared/static/qdmzn1esyjk6hwa4gy9y5y0tp0fpcr0n" -O spam_detection_training_data.csv

# The test images (without points)
!wget "https://sussex.box.com/shared/static/2a1am3esr4yzjmqr172vua1t41a846e0" -O spam_detection_test_data.csv

# Check the data downloaded correctly
If any of these assertions fail, redownload the data

In [ ]:
def confirm_checksum(filename, true_checksum):
  import subprocess
  checksum = subprocess.check_output(['shasum',filename]).decode('ascii')
  assert checksum.split(' ')[0] == true_checksum, 'Checksum does not match for ' + filename + ' redownload the data.'

confirm_checksum('spam_detection_training_data.csv', '807818a04b1f14412767e0929014fe0279047188')
confirm_checksum('spam_detection_test_data.csv', '93556bbad693968096613011355fa490d1fad4d5')



# Load the data

In [ ]:
import pandas as pd

# Load the data using np.load
data = pd.read_csv('spam_detection_training_data.csv')

# Extract the text
text = data['text'].values
# and the labels
labels = data['label'].values

print(text.shape, labels.shape)

test_data = pd.read_csv('spam_detection_test_data.csv')
test_text = test_data['text'].values
print(test_text.shape)

# Data Visualisation
Here's an example of how to display the text based on its label.

In [ ]:
def print_text(text, label):
  if label == 0:
    print (text, '\nis not spam!')
  else:
    print (text, '\nis spam!')

import numpy as np
idx = np.random.randint(0, text.shape[0])
print_text(text[idx], labels[idx])

# Calculating Confusion Matrix and exporting results

In [ ]:
from sklearn.metrics import confusion_matrix

def confusion_matrix(true_label, pred_label):
  """
  Calculate the confusion matrix for your predicted labels. See https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
  :param pred_label: Array of predicted labels
  :param true_label: Array of corresponding ground truth (test) labels
  :return: Confusion matrix whose i-th row and j-th column entry indicates the number of samples with true label being i-th class and predicted label being j-th class.
  """
  return confusion_matrix(true_label, pred_label)


In [ ]:
def save_as_csv(pred_labels, location = '.'):
    """
    Save the labels out as a .csv file
    :pred_labels: numpy array of shape (no_test_labels,) to be saved
    :param location: Directory to save results.csv in. Default to current working directory
    """
    assert pred_labels.shape[0]==1552, 'wrong number of labels, should be 1552 test labels'
    np.savetxt(location + '/results_task1.csv', pred_labels, delimiter=',')


In [ ]:
import seaborn as sns  # Used for creating attractive and informative statistical graphics
import matplotlib.pyplot as plt  # Used for plotting graphs and visualizing data
import re  # Provides regular expression matching operations for string manipulation
import numpy as np  # Supports large, multi-dimensional arrays and mathematical operations
import pandas as pd  # Provides data structures and data analysis tools, especially DataFrames
import joblib  # Enables saving and loading machine learning models efficiently
from sklearn.base import TransformerMixin  # Base class for all transformers in scikit-learn
from sklearn.pipeline import Pipeline  # Helps build a machine learning pipeline with steps
from sklearn.linear_model import LogisticRegression  # Logistic regression classifier
from sklearn.naive_bayes import MultinomialNB  # Naive Bayes classifier for multinomial models
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer  # Converts text to numeric feature vectors
from sklearn.model_selection import train_test_split, RandomizedSearchCV  # Splits data and performs parameter search with cross-validation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score  # Tools for evaluating model performance
from sklearn.preprocessing import FunctionTransformer  # Wraps a Python function into a transformer for pipelines


In [ ]:
# Remove missing entries and assign features and labels
cleaned_df = data.dropna()
texts = cleaned_df.loc[:, 'text']
labels = cleaned_df.loc[:, 'label']

In [ ]:
# Define a simple text normalization routine
def preprocess_text(batch):
    processed = []
    for line in batch:
        temp = re.sub(r'[^a-zA-Z0-9]', ' ', line)      # Replace non-alphanumeric characters with space
        temp = re.sub(r'\s+', ' ', temp).strip().lower()  # Collapse whitespace and convert to lowercase
        processed.append(temp)
    return processed

In [ ]:
# Partition the data into training and validation groups
X_train, X_valid, y_train, y_valid = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
# Container for storing results
evaluation_logs = []

# Updated model definitions and randomized hyperparameter spaces
classifier_configurations = {
    'BayesClassifier': {
        'workflow': Pipeline([
            ('text_cleaning', FunctionTransformer(preprocess_text, validate=False)),
            ('vectorizer', TfidfVectorizer(sublinear_tf=True)),
            ('model', MultinomialNB())
        ]),
        'search_space': {
            'vectorizer__ngram_range': [(1, 1), (2, 2)],
            'vectorizer__max_df': [0.85, 0.9],
            'vectorizer__min_df': [1, 3],
            'model__alpha': [0.5, 1.5, 3.0]
        }
    },
    'LogisticClassifier': {
        'workflow': Pipeline([
            ('text_cleaning', FunctionTransformer(preprocess_text, validate=False)),
            ('vectorizer', CountVectorizer(binary=True)),
            ('model', LogisticRegression(solver='liblinear'))
        ]),
        'search_space': {
            'vectorizer__ngram_range': [(1, 2), (2, 3)],
            'vectorizer__max_features': [1500, 2500, 4000],
            'model__C': [0.05, 0.5, 5.0],
            'model__penalty': ['l1', 'l2']
        }
    }
}


In [ ]:
def train_and_evaluate_model(classifier_configurations, X_train, y_train, X_valid, y_valid):
    """
    Trains and evaluates multiple models using RandomizedSearchCV, saves the best models, and visualizes results.

    Parameters:
    - classifier_configurations: Dictionary of models and their configurations (pipeline and parameter search space)
    - X_train: Training features
    - y_train: Training labels
    - X_valid: Validation features
    - y_valid: Validation labels
    """
    # Store evaluation results for all models
    evaluation_logs = []

    # Iterate over each model setup
    for model_label, config in classifier_configurations.items():
        print(f"\n>>> Starting training for: {model_label}")

        # Setup randomized search with 3-fold cross-validation
        random_search = RandomizedSearchCV(
            estimator=config['workflow'],
            param_distributions=config['search_space'],
            n_iter=5,
            cv=3,
            scoring='f1',
            verbose=1,
            random_state=42,
            n_jobs=-1
        )

        # Fit model to training data
        random_search.fit(X_train, y_train)

        # Save the trained best model to disk
        joblib.dump(random_search.best_estimator_, f"{model_label}_final_model.pkl")

        # Evaluate on validation set
        predictions = random_search.predict(X_valid)
        performance = {
            'Model': model_label,
            'Accuracy': accuracy_score(y_valid, predictions),
            'Precision': precision_score(y_valid, predictions),
            'Recall': recall_score(y_valid, predictions),
            'F1 Score': f1_score(y_valid, predictions)
        }
        evaluation_logs.append(performance)

        # Confusion matrix heatmap
        conf_mat = confusion_matrix(y_valid, predictions)
        fig, ax = plt.subplots()
        sns.heatmap(conf_mat, annot=True, cmap='crest', fmt='d', ax=ax)
        ax.set_title(f"{model_label} - Confusion Matrix")
        ax.set_xlabel("Predicted Labels")
        ax.set_ylabel("Actual Labels")
        plt.tight_layout()
        plt.show()

    return evaluation_logs

In [ ]:
evaluation_results = train_and_evaluate_model(classifier_configurations, X_train, y_train, X_valid, y_valid)

In [ ]:
results_df = pd.DataFrame(evaluation_results)
results_df

In [ ]:
# Load the previously trained Logistic Regression model from disk
loaded_model = joblib.load('LogRegModel_final_model.pkl')

# Apply the model to make predictions on the test set
test_predictions = loaded_model.predict(test_data['text'])

# Convert predictions to integers
test_predictions = test_predictions.astype(int)

# Store the predictions in a DataFrame and save as CSV
prediction_df = pd.DataFrame(test_predictions, columns=['Predicted_Label'])
prediction_df.to_csv('predictions_output.csv', index=False)

In [ ]:
# Generate predictions on validation dataset using the loaded pipeline
predictions_on_val = loaded_model.predict(X_valid)

# Create a DataFrame to hold email text, predicted labels, and actual labels
validation_results = pd.DataFrame({
    'Text': X_valid,
    'Predicted_Label': predictions_on_val.astype(int),
    'True_Label': y_valid
})

# Identify rows where predictions do not match the true labels
misclassified = validation_results[validation_results['Predicted_Label'] != validation_results['True_Label']]

# Label mapping for interpretation
label_interpretation = {0: 'Not Spam', 1: 'Spam'}

# Print the details of each misclassified email, but stop after two examples
misclassified_count = 0
for index, entry in misclassified.iterrows():
    if misclassified_count == 2:
        break  # Stop after 2 misclassified emails

    print(f"--- Misclassification {index} ---")
    print(f"Email Content: \n{entry['Text']}")
    print(f"True Label: {label_interpretation[entry['True_Label']]}")
    print(f"Predicted Label: {label_interpretation[entry['Predicted_Label']]}\n")

    misclassified_count += 1

